In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Postgres-Write") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.5.0") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/10 18:06:56 WARN Utils: Your hostname, codespaces-ce2559, resolves to a loopback address: 127.0.0.1; using 10.0.11.220 instead (on interface eth0)
26/04/10 18:06:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/workspaces/de-zoomcamp-final-project/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/codespace/.ivy2.5.2/cache
The jars for the packages stored in: /home/codespace/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f30d0fc8-301e-4ea8-83ab-dc0a34a4e416;1.0
	confs: [default]
	found org.postgresql#postgresql;42.5.0 in central
	found org.checkerframework#checker-qual;3.5.0 in central
downloading https://repo1.maven.org/maven2/org/postgresql/postgresql/42.5.0/postgresql

In [3]:
from pyspark.sql import types

In [4]:
schema = types.StructType([
    types.StructField('year', types.LongType(), True),
    types.StructField('Belgium', types.LongType(), True),
    types.StructField('France', types.LongType(), True),
    types.StructField('Germany', types.LongType(), True),
    types.StructField('Italy', types.LongType(), True),
    types.StructField('Poland', types.LongType(), True),
    types.StructField('Spain', types.LongType(), True)
])

In [5]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('./datalake/bronze/GDP_table.csv')

In [6]:
from pyspark.sql import functions as F

In [7]:
def get_live_usd_to_eur():
    """
    Fetches the real-time USD to EUR exchange rate using the Frankfurter API.
    Returns a float value of the rate, or a fallback value if the request fails.
    """
    try:
        # Endpoint for the latest exchange rates from USD to EUR
        url = "https://api.frankfurter.app/latest?from=USD&to=EUR"
        response = requests.get(url)
        data = response.json()
        
        # Extract the EUR rate from the response payload
        rate = data['rates']['EUR']
        print(f"Successfully fetched live rate: 1 USD = {rate} EUR")
        return rate
    except Exception as e:
        # Log the error and return a fallback rate to ensure pipeline continuity
        print(f"Failed to fetch exchange rate: {e}")
        return 0.92

In [8]:
# 1. Fetch the dynamic exchange rate via API
current_rate = get_live_usd_to_eur()

Failed to fetch exchange rate: name 'requests' is not defined


In [9]:
df_euro = df \
    .withColumn('Belgium', (F.col('Belgium') * current_rate).cast('long')) \
    .withColumn('France', (F.col('France') * current_rate).cast('long')) \
    .withColumn('Germany', (F.col('Germany') * current_rate).cast('long')) \
    .withColumn('Italy', (F.col('Italy') * current_rate).cast('long')) \
    .withColumn('Poland', (F.col('Poland') * current_rate).cast('long')) \
    .withColumn('Spain', (F.col('Spain') * current_rate).cast('long'))

In [10]:
df_euro.show()

+----+------------+-------------+-------------+-------------+------------+-------------+
|year|     Belgium|       France|      Germany|        Italy|      Poland|        Spain|
+----+------------+-------------+-------------+-------------+------------+-------------+
|2017|462543542911|2387538961582|3395581220316|1804852501565|484388167120|1207536137105|
|2018|499879578594|2567680328447|3659106298956|1924577832165|540418805549|1307314611111|
|2019|492546157495|2510560626969|3577260645536|1850382883147|549498119498|1281602405686|
|2020|479942786764|2419892312938|3538700814361|1741168139084|548894407262|1178965868840|
|2021|551888703347|2702474937316|3885266909491|1931889782398|620124405085|1311254459380|
+----+------------+-------------+-------------+-------------+------------+-------------+



In [11]:
df_euro = df_euro.repartition(24)

In [12]:
df_euro.write \
    .mode('overwrite') \
    .parquet('./datalake/silber/GDP_table.csv')

AnalysisException: [PATH_ALREADY_EXISTS] Path file:/workspaces/de-zoomcamp-final-project/datalake/silber/GDP_table.csv already exists. Set mode as "overwrite" to overwrite the existing path. SQLSTATE: 42K04

In [13]:
url = "jdbc:postgresql://localhost:5432/eu_gdp"
properties = {
    "user": "root",
    "password": "root",
    "driver": "org.postgresql.Driver"
}

In [14]:
df_euro.write.jdbc(
    url=url, 
    table="eu_gdp", 
    mode="overwrite", 
    properties=properties
)